In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

df = pd.read_csv("dataset_july_20.csv")

In [13]:
def cosine_similarity(a: np.ndarray, b: np.ndarray):
    norm_product = np.linalg.norm(a) * np.linalg.norm(b)
    if norm_product == 0: return 0.0
    return float(np.dot(a, b) / norm_product)
    
def cosine_similarity_matrix(a: np.ndarray, b: np.ndarray):
    a_norm = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_norm = b / np.linalg.norm(b, axis=1, keepdims=True)
    return a_norm @ b_norm.T

In [3]:
df.head()

,Timestamp,activity,category
0,7/20/2026 18:39:09,Bouldering,Sports
1,7/20/2026 18:39:11,I like reading webtoons,Reading/Books
2,7/20/2026 18:39:14,writing fiction,Reading/Books
3,7/20/2026 18:39:23,Voice Acting,Arts
4,7/20/2026 18:39:25,Hiking leads to a lot of fun and great scenic ...,Travel/Outdoors


In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8508.22it/s]


In [6]:
test = "I have been playing a lot of slay the spire 2, but I also like watching movies"

query_embedding = model.encode(test)

In [7]:
query_embedding

array([ 7.46441830e-04, -1.17444977e-01,  1.93057265e-02, -2.87953373e-02,
       -1.66335069e-02,  4.90937643e-02, -2.13475917e-02,  2.51526684e-02,
       -1.15175173e-02,  7.65543506e-02, -8.12005624e-02,  2.97904424e-02,
       -7.32403323e-02,  6.13542981e-02,  1.91979427e-02,  1.22904265e-03,
        8.32257122e-02, -1.01704989e-02,  4.04044203e-02, -1.78791359e-02,
       -2.60873437e-02, -8.27504694e-02,  3.34601314e-03,  3.68573004e-03,
        1.75228936e-03,  4.86323424e-03, -3.34928297e-02,  3.15295644e-02,
       -4.01305817e-02, -6.29237592e-02,  1.36301992e-02,  1.48234339e-02,
       -7.67658055e-02,  8.79065483e-04, -7.39687458e-02, -2.67362986e-02,
        6.17602691e-02, -1.31415948e-01, -1.48785889e-01, -4.97323759e-02,
        3.55797657e-03,  1.18162990e-01,  3.15009281e-02, -2.28327848e-02,
       -1.65702626e-02, -7.47936368e-02, -3.29869948e-02, -3.95417511e-02,
        4.77824584e-02,  4.39064465e-02,  3.56721282e-02, -5.51759964e-03,
       -7.19268993e-03, -

In [8]:
texts = df["activity"].to_list()

In [9]:
document_embeddings = model.encode(texts)

In [10]:
document_embeddings

array([[-3.83439176e-02,  5.66874631e-02,  7.01797083e-02, ...,
        -1.17707536e-01,  1.51362857e-02, -5.27941175e-02],
       [-6.61628246e-02, -8.11682120e-02, -6.96940273e-02, ...,
         5.39536588e-02,  2.69803940e-03,  3.52379680e-02],
       [ 1.15028806e-02, -2.45191809e-02,  8.07629153e-02, ...,
         1.54281582e-03,  9.47718229e-03, -5.48374839e-02],
       ...,
       [-1.71456598e-02, -6.96660876e-02,  4.54195365e-02, ...,
         1.34385088e-02,  1.54708931e-02, -4.21309024e-02],
       [ 4.87561785e-02,  1.40169039e-02,  5.52156453e-05, ...,
         5.00839576e-02, -9.86676812e-02, -3.93462181e-02],
       [ 2.98440224e-03, -3.53777199e-04, -2.60574128e-02, ...,
         7.22838417e-02,  1.22647993e-02, -3.91820706e-02]],
      shape=(27, 384), dtype=float32)

In [ ]:
sims = cosine_similarity_matrix(query_embedding.reshape(1, -1), document_embeddings)[0]

In [17]:
results = df.copy()
results["similarity"] = sims

results = results.sort_values("similarity", ascending=False)

In [18]:
results

,Timestamp,activity,category,similarity
11,7/20/2026 18:40:36,Watching movies,Arts,0.550615
8,7/20/2026 18:40:12,I like to play games.,Gaming,0.432254
7,7/20/2026 18:40:06,I enjoy watching TV shows and discussing them ...,Arts,0.380731
9,7/20/2026 18:40:22,I like playing gacha games without spending mo...,Gaming,0.360670
24,7/20/2026 18:44:50,I enjoy watching horrible Tubi movies just to ...,Arts,0.360657
4,7/20/2026 18:39:25,Hiking leads to a lot of fun and great scenic ...,Travel/Outdoors,0.346487
5,7/20/2026 18:39:39,Golfing is my favourite activity.,Sports,0.345188
25,7/20/2026 18:45:02,I like going camping often.,Travel/Outdoors,0.340112
18,7/20/2026 18:42:04,i like going hiking when I have free time,Travel/Outdoors,0.307109
16,7/20/2026 18:41:01,I like to play ice hockey in the fall.,Sports,0.290431
